# 06 — Controlled Feature-Set Experiments and Label-Rule Sensitivity

**Dataset:** Kaggle *Metaverse Financial Transactions Dataset*  
**Source:** https://www.kaggle.com/datasets/faizaniftikharjanjua/metaverse-financial-transactions-dataset  
**Expected data shape:** 78,600 rows × 14 columns  
**Target:** `anomaly` (`low_risk`, `moderate_risk`, `high_risk`)

## Purpose

This notebook compares carefully controlled feature sets using the same rows, chronological split, preprocessing, and model settings. It also tests whether conclusions change when the target is viewed under two explicitly different risk-grouping definitions.

This is an experimental sensitivity analysis. It does **not** prove the dataset's labels are ground truth, reconstruct the Kaggle creator's hidden rules, or establish real-world fraud detection.

## Important safeguards

- Keep the original CSV unchanged.
- `risk_score` is excluded from all predictive feature sets because previous audits found perfect separation of the target classes.
- Raw sender/receiver addresses are excluded from ordinary predictive inputs because they are repeated identifiers.
- `age_group` and `purchase_pattern` are not used together because previous results showed a one-to-one mapping.
- Suspected features are tested separately as controlled additions, not silently accepted as safe.
- No feature selection, imputation, scaling, or encoding is fitted on the test set.
- The original three-class target remains the primary task. Binary regroupings are sensitivity analyses only and must be reported separately.

## 1. Load data and verify source

In [1]:
from pathlib import Path
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = Path("metaverse_transactions_dataset.csv")
if not DATA_PATH.exists():
    # Accommodate the alternate filename used in the earlier upload.
    alternatives = [
        Path("metaverse_transactions_dataset(1).csv"),
        *Path(".").glob("*.csv")
    ]
    found = next((p for p in alternatives if p.exists()), None)
    if found is None:
        raise FileNotFoundError(
            "Upload the original Kaggle CSV to this Colab session, "
            "or edit DATA_PATH. No matching CSV was found."
        )
    DATA_PATH = found

file_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
df = pd.read_csv(DATA_PATH)

TARGET = "anomaly"
required = {TARGET, "timestamp"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {sorted(missing_required)}")

print("Source file:", DATA_PATH.name)
print("SHA-256:", file_sha256)
print("Shape:", df.shape)
print("Target counts:")
display(df[TARGET].value_counts(dropna=False).to_frame("count"))

Source file: metaverse_transactions_dataset.csv
SHA-256: c7eae68df7ff627089b44df6123c4974e71065f4c342b45a4bf4664bdea6beb0
Shape: (78600, 14)
Target counts:


,count
anomaly,
low_risk,63494
moderate_risk,8611
high_risk,6495


## 2. Define the primary chronological split

In [2]:
# Parse timestamps. Invalid timestamps are excluded from the chronological
# experiment and reported; do not silently impute their time order.
work = df.copy()
work["_parsed_timestamp"] = pd.to_datetime(work["timestamp"], errors="coerce", utc=True)
invalid_time = int(work["_parsed_timestamp"].isna().sum())
print("Rows with invalid timestamps:", invalid_time)

work = work.loc[work["_parsed_timestamp"].notna()].copy()
work = work.sort_values("_parsed_timestamp", kind="mergesort").reset_index(drop=True)

# Primary split: earliest 80% for training, latest 20% for testing.
cut = int(len(work) * 0.80)
train_df = work.iloc[:cut].copy()
test_df = work.iloc[cut:].copy()

print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Train period:", train_df["_parsed_timestamp"].min(), "to", train_df["_parsed_timestamp"].max())
print("Test period: ", test_df["_parsed_timestamp"].min(), "to", test_df["_parsed_timestamp"].max())

print("\nTraining class distribution:")
display(train_df[TARGET].value_counts(normalize=True).mul(100).round(2).to_frame("percent"))
print("Testing class distribution:")
display(test_df[TARGET].value_counts(normalize=True).mul(100).round(2).to_frame("percent"))

if set(train_df[TARGET].unique()) != set(test_df[TARGET].unique()):
    warnings.warn("Train and test do not contain the same target classes. Review before interpreting model metrics.")

Rows with invalid timestamps: 0
Train rows: 62880 | Test rows: 15720
Train period: 2022-01-01 00:00:45+00:00 to 2022-10-20 17:47:26+00:00
Test period:  2022-10-20 17:50:42+00:00 to 2022-12-31 23:47:40+00:00

Training class distribution:


,percent
anomaly,
low_risk,80.71
moderate_risk,10.97
high_risk,8.32


Testing class distribution:


,percent
anomaly,
low_risk,81.06
moderate_risk,10.89
high_risk,8.05


## 3. Build transparent feature sets

The feature sets are nested or controlled additions. This design helps show how model performance changes when a suspicious feature group is introduced.

| Set | Included fields | Interpretation |
|---|---|---|
| FS0 — Minimal operational | `amount`, `hour_of_day` | Minimal numeric/time baseline |
| FS1 — User activity added | FS0 + `login_frequency`, `session_duration` | Tests activity signals |
| FS2 — Context added | FS1 + `location_region`, `ip_prefix` | Tests context signals |
| FS3 — Transaction type added | FS2 + `transaction_type` | Tests a strongly target-associated field |
| FS4 — Age group added | FS2 + `age_group` | Tests one of the redundant demographic/behavioral fields |
| FS5 — Purchase pattern added | FS2 + `purchase_pattern` | Tests the alternative redundant field |

`risk_score`, raw addresses, raw timestamp strings, and the target are excluded from every feature set. The timestamp is used only for ordering. If a column is absent, that feature set is skipped with a warning.

In [3]:
feature_sets_requested = {
    "FS0_minimal_operational": ["amount", "hour_of_day"],
    "FS1_plus_user_activity": ["amount", "hour_of_day", "login_frequency", "session_duration"],
    "FS2_plus_context": ["amount", "hour_of_day", "login_frequency", "session_duration",
                        "location_region", "ip_prefix"],
    "FS3_plus_transaction_type": ["amount", "hour_of_day", "login_frequency", "session_duration",
                                  "location_region", "ip_prefix", "transaction_type"],
    "FS4_plus_age_group": ["amount", "hour_of_day", "login_frequency", "session_duration",
                           "location_region", "ip_prefix", "age_group"],
    "FS5_plus_purchase_pattern": ["amount", "hour_of_day", "login_frequency", "session_duration",
                                  "location_region", "ip_prefix", "purchase_pattern"],
}

feature_sets = {}
for name, cols in feature_sets_requested.items():
    available = [c for c in cols if c in work.columns]
    missing = [c for c in cols if c not in work.columns]
    if missing:
        print(f"Skipping missing columns for {name}: {missing}")
    if available:
        feature_sets[name] = available

for name, cols in feature_sets.items():
    print(f"{name}: {cols}")

FS0_minimal_operational: ['amount', 'hour_of_day']
FS1_plus_user_activity: ['amount', 'hour_of_day', 'login_frequency', 'session_duration']
FS2_plus_context: ['amount', 'hour_of_day', 'login_frequency', 'session_duration', 'location_region', 'ip_prefix']
FS3_plus_transaction_type: ['amount', 'hour_of_day', 'login_frequency', 'session_duration', 'location_region', 'ip_prefix', 'transaction_type']
FS4_plus_age_group: ['amount', 'hour_of_day', 'login_frequency', 'session_duration', 'location_region', 'ip_prefix', 'age_group']
FS5_plus_purchase_pattern: ['amount', 'hour_of_day', 'login_frequency', 'session_duration', 'location_region', 'ip_prefix', 'purchase_pattern']


## 4. Define fixed models and leakage-safe preprocessing

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

def make_onehot():
    # Compatibility with both newer and older scikit-learn releases.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def build_pipeline(X_train, model_name, random_state=42):
    numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot())
    ])
    preprocess = ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", categorical_pipe, categorical_cols)
    ])

    if model_name == "Dummy":
        estimator = DummyClassifier(strategy="most_frequent")
    elif model_name == "LogisticRegression":
        estimator = LogisticRegression(
            max_iter=2000, class_weight="balanced",
            random_state=random_state
        )
    elif model_name == "RandomForest":
        estimator = RandomForestClassifier(
            n_estimators=250, min_samples_leaf=3,
            class_weight="balanced_subsample",
            random_state=random_state, n_jobs=-1
        )
    else:
        raise ValueError(model_name)

    return Pipeline([("preprocess", preprocess), ("model", estimator)])

## 5. Run controlled three-class experiments

In [5]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, classification_report,
    confusion_matrix, log_loss
)

MODEL_NAMES = ["Dummy", "LogisticRegression", "RandomForest"]
y_train = train_df[TARGET].astype(str)
y_test = test_df[TARGET].astype(str)
primary_labels = sorted(df[TARGET].dropna().astype(str).unique())

experiment_rows = []
confusion_outputs = {}
report_outputs = {}

for fs_name, cols in feature_sets.items():
    X_train = train_df[cols].copy()
    X_test = test_df[cols].copy()

    for model_name in MODEL_NAMES:
        print(f"Running {fs_name} | {model_name}")
        pipe = build_pipeline(X_train, model_name)
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

        row = {
            "feature_set": fs_name,
            "model": model_name,
            "n_features": len(cols),
            "features": ", ".join(cols),
            "accuracy": accuracy_score(y_test, pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
            "macro_precision": precision_score(y_test, pred, average="macro", zero_division=0),
            "macro_recall": recall_score(y_test, pred, average="macro", zero_division=0),
            "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0),
        }
        if hasattr(pipe, "predict_proba"):
            try:
                probs = pipe.predict_proba(X_test)
                # Align probability columns to estimator's class ordering.
                row["log_loss"] = log_loss(y_test, probs, labels=pipe.classes_)
            except Exception:
                row["log_loss"] = np.nan
        else:
            row["log_loss"] = np.nan

        experiment_rows.append(row)
        cm = confusion_matrix(y_test, pred, labels=primary_labels)
        confusion_outputs[(fs_name, model_name)] = pd.DataFrame(
            cm, index=primary_labels, columns=primary_labels
        )
        report_outputs[(fs_name, model_name)] = pd.DataFrame(
            classification_report(y_test, pred, labels=primary_labels,
                                  output_dict=True, zero_division=0)
        ).T

results = pd.DataFrame(experiment_rows).sort_values(
    ["macro_f1", "balanced_accuracy"], ascending=False
).reset_index(drop=True)
display(results)

Running FS0_minimal_operational | Dummy
Running FS0_minimal_operational | LogisticRegression
Running FS0_minimal_operational | RandomForest
Running FS1_plus_user_activity | Dummy
Running FS1_plus_user_activity | LogisticRegression
Running FS1_plus_user_activity | RandomForest
Running FS2_plus_context | Dummy
Running FS2_plus_context | LogisticRegression
Running FS2_plus_context | RandomForest
Running FS3_plus_transaction_type | Dummy
Running FS3_plus_transaction_type | LogisticRegression
Running FS3_plus_transaction_type | RandomForest
Running FS4_plus_age_group | Dummy
Running FS4_plus_age_group | LogisticRegression
Running FS4_plus_age_group | RandomForest
Running FS5_plus_purchase_pattern | Dummy
Running FS5_plus_purchase_pattern | LogisticRegression
Running FS5_plus_purchase_pattern | RandomForest


,feature_set,model,n_features,features,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,log_loss
0,FS3_plus_transaction_type,RandomForest,7,"amount, hour_of_day, login_frequency, session_...",1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.006628
1,FS3_plus_transaction_type,LogisticRegression,7,"amount, hour_of_day, login_frequency, session_...",0.931997,0.962428,0.871904,0.962428,0.904100,0.938089,0.155127
2,FS1_plus_user_activity,RandomForest,4,"amount, hour_of_day, login_frequency, session_...",0.807252,0.724733,0.616038,0.724733,0.655773,0.820983,0.354624
3,FS4_plus_age_group,RandomForest,7,"amount, hour_of_day, login_frequency, session_...",0.806361,0.724266,0.616046,0.724266,0.655500,0.820612,0.353945
4,FS5_plus_purchase_pattern,RandomForest,7,"amount, hour_of_day, login_frequency, session_...",0.804962,0.721288,0.613793,0.721288,0.652952,0.819272,0.353583
5,FS2_plus_context,RandomForest,6,"amount, hour_of_day, login_frequency, session_...",0.808461,0.717947,0.613465,0.717947,0.652219,0.820997,0.350068
6,FS4_plus_age_group,LogisticRegression,7,"amount, hour_of_day, login_frequency, session_...",0.629517,0.819821,0.565059,0.819821,0.571887,0.674977,0.661771
7,FS5_plus_purchase_pattern,LogisticRegression,7,"amount, hour_of_day, login_frequency, session_...",0.629517,0.819821,0.565059,0.819821,0.571887,0.674977,0.661771
8,FS2_plus_context,LogisticRegression,6,"amount, hour_of_day, login_frequency, session_...",0.599809,0.793461,0.539336,0.793461,0.540173,0.644423,0.745259
9,FS1_plus_user_activity,LogisticRegression,4,"amount, hour_of_day, login_frequency, session_...",0.598282,0.792259,0.538548,0.792259,0.538831,0.642998,0.745198


## 6. Inspect confusion matrices and per-class recall

For this risk task, `high_risk` recall is particularly important. Do not select a model based on accuracy alone. A model can achieve high accuracy by predicting the majority `low_risk` class while missing high-risk cases.

In [6]:
for key, cm_df in confusion_outputs.items():
    print("\\n", key)
    display(cm_df)

# Summarize class-specific recall from classification reports.
recall_rows = []
for (fs_name, model_name), report_df in report_outputs.items():
    for cls in primary_labels:
        if cls in report_df.index:
            recall_rows.append({
                "feature_set": fs_name,
                "model": model_name,
                "class": cls,
                "precision": report_df.loc[cls, "precision"],
                "recall": report_df.loc[cls, "recall"],
                "f1": report_df.loc[cls, "f1-score"],
                "support": report_df.loc[cls, "support"],
            })
class_metrics = pd.DataFrame(recall_rows)
display(class_metrics.pivot_table(
    index=["feature_set", "model"], columns="class", values="recall"
).round(4))

\n ('FS0_minimal_operational', 'Dummy')


,high_risk,low_risk,moderate_risk
high_risk,0,1266,0
low_risk,0,12742,0
moderate_risk,0,1712,0


\n ('FS0_minimal_operational', 'LogisticRegression')


,high_risk,low_risk,moderate_risk
high_risk,211,617,438
low_risk,2130,6786,3826
moderate_risk,301,229,1182


\n ('FS0_minimal_operational', 'RandomForest')


,high_risk,low_risk,moderate_risk
high_risk,162,860,244
low_risk,1619,9039,2084
moderate_risk,203,695,814


... (27 outputs left out)

,high_risk,low_risk,moderate_risk
high_risk,1256,0,10
low_risk,3878,7079,1785
moderate_risk,70,81,1561


\n ('FS5_plus_purchase_pattern', 'RandomForest')


,high_risk,low_risk,moderate_risk
high_risk,468,773,25
low_risk,1413,10529,800
moderate_risk,45,10,1657


class                                         high_risk  low_risk  \
feature_set               model                                     
FS0_minimal_operational   Dummy                  0.0000    1.0000   
                          LogisticRegression     0.1667    0.5326   
                          RandomForest           0.1280    0.7094   
FS1_plus_user_activity    Dummy                  0.0000    1.0000   
                          LogisticRegression     0.9779    0.5233   
                          RandomForest           0.3752    0.8282   
FS2_plus_context          Dummy                  0.0000    1.0000   
                          LogisticRegression     0.9787    0.5250   
                          RandomForest           0.3515    0.8321   
FS3_plus_transaction_type Dummy                  0.0000    1.0000   
                          LogisticRegression     1.0000    0.9206   
                          RandomForest           1.0000    1.0000   
FS4_plus_age_group        Dummy                  0.0000    1.0000   
                          LogisticRegression     0.9921    0.5556   
                          RandomForest           0.3760    0.8272   
FS5_plus_purchase_pattern Dummy                  0.0000    1.0000   
                          LogisticRegression     0.9921    0.5556   
                          RandomForest           0.3697    0.8263   

class                                         moderate_risk  
feature_set               model                              
FS0_minimal_operational   Dummy                      0.0000  
                          LogisticRegression         0.6904  
                          RandomForest               0.4755  
FS1_plus_user_activity    Dummy                      0.0000  
                          LogisticRegression         0.8756  
                          RandomForest               0.9708  
FS2_plus_context          Dummy                      0.0000  
                          LogisticRegression         0.8768  
                          RandomForest               0.9702  
FS3_plus_transaction_type Dummy                      0.0000  
                          LogisticRegression         0.9667  
                          RandomForest               1.0000  
FS4_plus_age_group        Dummy                      0.0000  
                          LogisticRegression         0.9118  
                          RandomForest               0.9696  
FS5_plus_purchase_pattern Dummy                      0.0000  
                          LogisticRegression         0.9118  
                          RandomForest               0.9679

## 7. Feature-set sensitivity: measure the incremental effect of suspicious fields

This table compares each feature set with FS2 (activity + context). A performance increase after adding a suspicious feature is **not** proof that the feature is legitimate. It may indicate that the field carries information used by the label-generation process.

In [7]:
comparison = results.pivot(index="feature_set", columns="model", values=["macro_f1", "balanced_accuracy"])
display(comparison.round(4))

base_name = "FS2_plus_context"
if base_name in set(results["feature_set"]):
    base = results[results["feature_set"] == base_name].set_index("model")
    delta_rows = []
    for fs_name in results["feature_set"].unique():
        current = results[results["feature_set"] == fs_name].set_index("model")
        for model_name in current.index:
            if model_name in base.index:
                delta_rows.append({
                    "feature_set": fs_name,
                    "model": model_name,
                    "delta_macro_f1_vs_FS2": current.loc[model_name, "macro_f1"] - base.loc[model_name, "macro_f1"],
                    "delta_balanced_accuracy_vs_FS2": current.loc[model_name, "balanced_accuracy"] - base.loc[model_name, "balanced_accuracy"],
                })
    display(pd.DataFrame(delta_rows).round(4))
else:
    print("FS2 not available; cannot calculate the planned reference comparison.")

macro_f1                                  \
model                        Dummy LogisticRegression RandomForest   
feature_set                                                          
FS0_minimal_operational     0.2985             0.3681       0.4033   
FS1_plus_user_activity      0.2985             0.5388       0.6558   
FS2_plus_context            0.2985             0.5402       0.6522   
FS3_plus_transaction_type   0.2985             0.9041       1.0000   
FS4_plus_age_group          0.2985             0.5719       0.6555   
FS5_plus_purchase_pattern   0.2985             0.5719       0.6530   

                          balanced_accuracy                                  
model                                 Dummy LogisticRegression RandomForest  
feature_set                                                                  
FS0_minimal_operational              0.3333             0.4632       0.4376  
FS1_plus_user_activity               0.3333             0.7923       0.7247  
FS2_plus_context                     0.3333             0.7935       0.7179  
FS3_plus_transaction_type            0.3333             0.9624       1.0000  
FS4_plus_age_group                   0.3333             0.8198       0.7243  
FS5_plus_purchase_pattern            0.3333             0.8198       0.7213

,feature_set,model,delta_macro_f1_vs_FS2,delta_balanced_accuracy_vs_FS2
0,FS3_plus_transaction_type,RandomForest,0.3478,0.2821
1,FS3_plus_transaction_type,LogisticRegression,0.3639,0.1690
2,FS3_plus_transaction_type,Dummy,0.0000,0.0000
3,FS1_plus_user_activity,RandomForest,0.0036,0.0068
4,FS1_plus_user_activity,LogisticRegression,-0.0013,-0.0012
5,FS1_plus_user_activity,Dummy,0.0000,0.0000
6,FS4_plus_age_group,RandomForest,0.0033,0.0063
7,FS4_plus_age_group,LogisticRegression,0.0317,0.0264
8,FS4_plus_age_group,Dummy,0.0000,0.0000
9,FS5_plus_purchase_pattern,RandomForest,0.0007,0.0033


## 8. Label-rule sensitivity analyses (separate tasks)

The original three-class `anomaly` target remains unchanged and is the primary task. The following two binary regroupings answer different questions and must not be mixed with the three-class results:

- **High-risk screening:** `high_risk` versus `low_risk` + `moderate_risk`.
- **Elevated-risk screening:** `low_risk` versus `moderate_risk` + `high_risk`.

These regroupings test sensitivity to the operational definition of a positive class. They do not verify whether the original labels are correct. They are not substitutes for independently verified outcomes.

In [8]:
from sklearn.metrics import average_precision_score, roc_auc_score

def binary_sensitivity(target_name, positive_label, feature_name="FS2_plus_context"):
    if feature_name not in feature_sets:
        print("Feature set unavailable:", feature_name)
        return None

    cols = feature_sets[feature_name]
    ytr_orig = train_df[TARGET].astype(str)
    yte_orig = test_df[TARGET].astype(str)

    if target_name == "high_risk_vs_rest":
        ytr = (ytr_orig == "high_risk").astype(int)
        yte = (yte_orig == "high_risk").astype(int)
    elif target_name == "elevated_risk_vs_low":
        ytr = ytr_orig.isin(["moderate_risk", "high_risk"]).astype(int)
        yte = yte_orig.isin(["moderate_risk", "high_risk"]).astype(int)
    else:
        raise ValueError(target_name)

    Xtr = train_df[cols].copy()
    Xte = test_df[cols].copy()
    out = []

    for model_name in ["Dummy", "LogisticRegression", "RandomForest"]:
        pipe = build_pipeline(Xtr, model_name)
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        row = {
            "target_definition": target_name,
            "feature_set": feature_name,
            "model": model_name,
            "positive_class": positive_label,
            "accuracy": accuracy_score(yte, pred),
            "balanced_accuracy": balanced_accuracy_score(yte, pred),
            "positive_precision": precision_score(yte, pred, zero_division=0),
            "positive_recall": recall_score(yte, pred, zero_division=0),
            "positive_f1": f1_score(yte, pred, zero_division=0),
        }
        if hasattr(pipe, "predict_proba") and len(np.unique(yte)) == 2:
            prob = pipe.predict_proba(Xte)[:, list(pipe.classes_).index(1)]
            row["average_precision"] = average_precision_score(yte, prob)
            row["roc_auc"] = roc_auc_score(yte, prob)
        out.append(row)
    return pd.DataFrame(out)

binary_results = pd.concat([
    binary_sensitivity("high_risk_vs_rest", "high_risk"),
    binary_sensitivity("elevated_risk_vs_low", "moderate_risk + high_risk")
], ignore_index=True)
display(binary_results.round(4))

,target_definition,feature_set,model,positive_class,accuracy,balanced_accuracy,positive_precision,positive_recall,positive_f1,average_precision,roc_auc
0,high_risk_vs_rest,FS2_plus_context,Dummy,high_risk,0.9195,0.5000,0.0000,0.0000,0.0000,0.0805,0.5000
1,high_risk_vs_rest,FS2_plus_context,LogisticRegression,high_risk,0.7469,0.8617,0.2412,0.9984,0.3886,0.2517,0.8670
2,high_risk_vs_rest,FS2_plus_context,RandomForest,high_risk,0.8614,0.6378,0.2536,0.3712,0.3014,0.2520,0.8668
3,elevated_risk_vs_low,FS2_plus_context,Dummy,moderate_risk + high_risk,0.8106,0.5000,0.0000,0.0000,0.0000,0.1894,0.5000
4,elevated_risk_vs_low,FS2_plus_context,LogisticRegression,moderate_risk + high_risk,0.6259,0.6291,0.2827,0.6343,0.3911,0.3549,0.6802
5,elevated_risk_vs_low,FS2_plus_context,RandomForest,moderate_risk + high_risk,0.8407,0.7718,0.5685,0.6608,0.6112,0.6108,0.8938


## 9. Temporal-cutoff sensitivity (same feature set and models)

A single chronological cutoff can produce a result that depends on the chosen time window. This section repeats the reference feature set using 70/30, 80/20, and 90/10 chronological splits. It does not tune thresholds or hyperparameters.

The 80/20 result should match the primary run apart from minor implementation differences. If any split lacks a class in training or test, the relevant metrics must be interpreted cautiously.

In [9]:
def evaluate_time_split(train_fraction, feature_name="FS2_plus_context"):
    cols = feature_sets[feature_name]
    cut_local = int(len(work) * train_fraction)
    tr = work.iloc[:cut_local]
    te = work.iloc[cut_local:]
    if tr.empty or te.empty:
        return []

    ytr = tr[TARGET].astype(str)
    yte = te[TARGET].astype(str)
    rows = []
    for model_name in ["Dummy", "LogisticRegression", "RandomForest"]:
        Xtr, Xte = tr[cols].copy(), te[cols].copy()
        pipe = build_pipeline(Xtr, model_name)
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        rows.append({
            "train_fraction": train_fraction,
            "model": model_name,
            "train_rows": len(tr),
            "test_rows": len(te),
            "train_start": tr["_parsed_timestamp"].min(),
            "train_end": tr["_parsed_timestamp"].max(),
            "test_start": te["_parsed_timestamp"].min(),
            "test_end": te["_parsed_timestamp"].max(),
            "balanced_accuracy": balanced_accuracy_score(yte, pred),
            "macro_f1": f1_score(yte, pred, average="macro", zero_division=0),
            "high_risk_recall": recall_score(
                yte, pred, labels=["high_risk"], average="macro", zero_division=0
            ) if "high_risk" in set(yte) else np.nan,
        })
    return rows

temporal_rows = []
for frac in [0.70, 0.80, 0.90]:
    temporal_rows.extend(evaluate_time_split(frac))

temporal_results = pd.DataFrame(temporal_rows)
display(temporal_results)

,train_fraction,model,train_rows,test_rows,train_start,train_end,test_start,test_end,balanced_accuracy,macro_f1,high_risk_recall
0,0.7,Dummy,55020,23580,2022-01-01 00:00:45+00:00,2022-09-13 23:22:23+00:00,2022-09-13 23:23:38+00:00,2022-12-31 23:47:40+00:00,0.333333,0.298594,0.000000
1,0.7,LogisticRegression,55020,23580,2022-01-01 00:00:45+00:00,2022-09-13 23:22:23+00:00,2022-09-13 23:23:38+00:00,2022-12-31 23:47:40+00:00,0.794950,0.543819,0.979365
2,0.7,RandomForest,55020,23580,2022-01-01 00:00:45+00:00,2022-09-13 23:22:23+00:00,2022-09-13 23:23:38+00:00,2022-12-31 23:47:40+00:00,0.725458,0.659462,0.379894
3,0.8,Dummy,62880,15720,2022-01-01 00:00:45+00:00,2022-10-20 17:47:26+00:00,2022-10-20 17:50:42+00:00,2022-12-31 23:47:40+00:00,0.333333,0.298456,0.000000
4,0.8,LogisticRegression,62880,15720,2022-01-01 00:00:45+00:00,2022-10-20 17:47:26+00:00,2022-10-20 17:50:42+00:00,2022-12-31 23:47:40+00:00,0.793461,0.540173,0.978673
5,0.8,RandomForest,62880,15720,2022-01-01 00:00:45+00:00,2022-10-20 17:47:26+00:00,2022-10-20 17:50:42+00:00,2022-12-31 23:47:40+00:00,0.717947,0.652219,0.351501
6,0.9,Dummy,70740,7860,2022-01-01 00:00:45+00:00,2022-11-25 16:59:27+00:00,2022-11-25 17:08:02+00:00,2022-12-31 23:47:40+00:00,0.333333,0.298612,0.000000
7,0.9,LogisticRegression,70740,7860,2022-01-01 00:00:45+00:00,2022-11-25 16:59:27+00:00,2022-11-25 17:08:02+00:00,2022-12-31 23:47:40+00:00,0.793315,0.541681,0.981191
8,0.9,RandomForest,70740,7860,2022-01-01 00:00:45+00:00,2022-11-25 16:59:27+00:00,2022-11-25 17:08:02+00:00,2022-12-31 23:47:40+00:00,0.730138,0.660781,0.384013


## 10. Export experiment results and reproducibility metadata

The CSV files below are the outputs to upload with the completed notebook. They preserve feature-set definitions and the source file hash.

In [10]:
from pathlib import Path
Path("outputs_06").mkdir(exist_ok=True)

results.to_csv("outputs_06/06_three_class_feature_set_results.csv", index=False)
class_metrics.to_csv("outputs_06/06_three_class_per_class_metrics.csv", index=False)
binary_results.to_csv("outputs_06/06_binary_label_sensitivity_results.csv", index=False)
temporal_results.to_csv("outputs_06/06_temporal_split_sensitivity_results.csv", index=False)
register_rows = []
for name, cols in feature_sets.items():
    for col in cols:
        register_rows.append({"feature_set": name, "feature": col})
pd.DataFrame(register_rows).to_csv("outputs_06/06_feature_set_definitions.csv", index=False)

metadata = {
    "source_file": DATA_PATH.name,
    "source_sha256": file_sha256,
    "rows_loaded": int(len(df)),
    "columns_loaded": int(df.shape[1]),
    "target": TARGET,
    "primary_split": "chronological 80/20",
    "feature_sets": feature_sets,
    "excluded_from_all_predictive_sets": [
        "risk_score", "sending_address", "receiving_address", "timestamp",
        "purchase_pattern when age_group is included in FS4",
        "age_group when purchase_pattern is included in FS5"
    ],
    "models": MODEL_NAMES,
    "note": "Binary target regroupings are sensitivity analyses, not replacements for the original three-class target."
}
Path("outputs_06/06_experiment_metadata.json").write_text(json.dumps(metadata, indent=2, default=str))

print("Saved all Notebook 06 outputs in outputs_06/")

Saved all Notebook 06 outputs in outputs_06/


## 11. Interpretation and reporting template

When interpreting results, answer all of the following:

1. Does FS0 provide meaningful discrimination beyond the majority-class baseline?
2. How much do `login_frequency` and `session_duration` change macro-F1 and high-risk recall when added in FS1?
3. Do context fields improve performance in FS2?
4. Does adding `transaction_type` in FS3 cause a large performance jump? If so, treat that as a provenance warning, not automatic evidence of a better model.
5. Do FS4 and FS5 behave similarly? Their near-equivalence would be consistent with the observed one-to-one mapping between `age_group` and `purchase_pattern`.
6. Does the high-risk screening task tell a different story from the elevated-risk screening task?
7. Are the conclusions stable across 70/30, 80/20, and 90/10 chronological cutoffs?
8. Does the model detect high-risk transactions with adequate recall, or does accuracy hide missed high-risk cases?

### Reporting limitations

- These are predictive associations with the Kaggle `anomaly` field, not validated fraud detection outcomes.
- The notebook cannot prove label-generation rules without source documentation or generator code.
- A feature that improves scores may be a proxy for how the labels were assigned.
- A lower score after excluding suspicious fields may be more scientifically credible than a perfect score driven by a label proxy.
- Do not select the final model solely by macro-F1. Consider high-risk recall, precision, false negatives, and the intended operational use.

## 12. Decision gate before the next ECIPS stage

- [ ] Review all feature-set results and confusion matrices.
- [ ] Identify which feature additions cause abrupt performance changes.
- [ ] Confirm feature availability at the prediction point.
- [ ] Keep `risk_score` excluded unless its independent provenance is documented.
- [ ] Decide whether the task is three-class risk classification, high-risk screening, or elevated-risk screening.
- [ ] State clearly that the target is Kaggle-provided risk labels unless independent outcome validation is obtained.
- [ ] Do not describe the best score as real-world fraud-detection performance without independent ground truth.

**Do not modify the original CSV or overwrite any previous notebook outputs.**